
# Différentes façon d'obtenir la taxonomie

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
from natsort import natsorted
from tqdm import tqdm
import requests
import yaml
from Bio import SeqIO, Entrez
from io import StringIO

sys.path.append("..")

from dataset.bactero_set import BacteriaDataset, TAXO_LEVELS

# Méthode I : par l'acccesion  (FAIT par SIEGFRIED)
1. On part de l'assembly summary ou seulement les génomes complets ont été retenus
2. à partir des fichiers télécharger, on récupère l'accession
3. on fait une requete sur la db 'nucleotide'
4. on récupère une liste 'taxonomy' à réodonner éventuellement et une autre 'organism'


In [191]:
completed_df  = pd.read_csv('assembly_summary_filtered_Complete_Genome.csv', sep='\t')
completed_df

,assembly_accession,refseq_category,taxid,species_taxid,ftp_path
0,GCF_900128725.1,na,9,9,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/9...
1,GCF_003044255.1,na,24,24,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...
2,GCF_009730575.1,na,24,24,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...
3,GCF_016406305.1,na,24,24,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...
4,GCF_016406325.1,reference genome,24,24,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...
...,...,...,...,...,...
47362,GCF_047758675.1,na,3398357,3398357,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...
47363,GCF_047758685.1,na,3398358,3398358,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...
47364,GCF_047777955.1,na,3398393,3398393,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...
47365,GCF_047824785.1,na,3398703,3398703,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/0...


In [192]:
def add_accesion_from_file(df_in, output_dir):
    # Ajouter une nouvelle colonne ou mettre à jour la colonne Downloaded
    df = df_in.copy()
    for index, row in tqdm(df.iterrows()):
        # Extraire l'URL FTP et créer le chemin du fichier attendu
        ftp_path = row['ftp_path']
        end_url_file = ftp_path[8:].split('/')[-1]  # On enlève 'https://' et on prend la dernière partie de l'URL
        file_path = os.path.join(output_dir, f"{end_url_file}_genomic.fna.gz")
        unzip_file_path = file_path.removesuffix(".gz")


        # Vérifier si le fichier .fna.gz existe
        if os.path.isfile(unzip_file_path):
            df.at[index, 'file'] = os.path.basename(unzip_file_path)
            df.at[index, 'Downloaded'] = True
            with open(unzip_file_path, "r", encoding='utf-8') as reader:
                # first_line = reader.readline().strip()
                first_line = reader.readline()
                accession = first_line.split('.')[0][1:]  # Extraction de l'accession
                df.at[index, 'accession']  = accession
        else:
            df.at[index, 'Downloaded'] = False
    return df

In [ ]:
output_dir = '/home/hcourtei/Projects/MicroTaxo/codes/wisp/wisp_light/import_dataset/refseq/out_refseq'
#output_dir = '/projects/microtaxo/data/refseq2'
df_with_accession = add_accesion_from_file(completed_df, output_dir)
print(df_with_accession[380:420].drop(columns=['ftp_path']).to_markdown())
df_with_accession.to_csv('Complete_Genome_with_accession.csv', sep='\t')


47367it [00:01, 36006.51it/s]

|     | assembly_accession   | refseq_category   |   taxid |   species_taxid | file                                     | Downloaded   | accession   |
|----:|:---------------------|:------------------|--------:|----------------:|:-----------------------------------------|:-------------|:------------|
| 380 | GCF_015245165.1      | na                |     197 |             197 | GCF_015245165.1_ASM1524516v1_genomic.fna | True         | NZ_CP063357 |
| 381 | GCF_016350125.1      | na                |     197 |             197 | GCF_016350125.1_ASM1635012v1_genomic.fna | True         | NZ_CP066242 |
| 382 | GCF_016458845.1      | na                |     197 |             197 | GCF_016458845.1_ASM1645884v1_genomic.fna | True         | NZ_CP066746 |
| 383 | GCF_016458865.1      | na                |     197 |             197 | GCF_016458865.1_ASM1645886v1_genomic.fna | True         | NZ_CP066745 |
| 384 | GCF_016600925.1      | na                |     197 |             197 | GCF_016600925.1

In [ ]:

accessions_df = df_with_accession[['accession']].dropna().reset_index(drop=True)
accessions_df


,accession
0,NZ_LT667500
1,NZ_CP028435
2,NZ_CP046329
3,NZ_CP066369
4,NZ_CP066370
...,...
385,NZ_CP071580
386,NZ_CP071581
387,NZ_CP071583
388,NZ_CP071584


In [ ]:
# https://biopython.org/docs/1.76/api/Bio.Entrez.html
def get_taxo_by_accession(accession_df):
    
    Entrez.email = "hermann.courteille@inria.fr"
    Entrez.api_key =  "b55513ab1634ec527ccf1ec084f3b1c78108"
    Entrez.max_tries = 5
    Entrez.sleep_between_tries = 15
    batchsize= 5
    result_df = accession_df.copy()

    for start in tqdm(range(0, len(accession_df), batchsize)):
        accession_df_batch = accession_df[start:start + batchsize].copy()

        with Entrez.efetch(db="nucleotide", id=accession_df_batch.accession.to_list(), rettype="gb", retmode="text") as taxo_handle:
            records = SeqIO.parse(taxo_handle, 'genbank')
            for idx , record in enumerate(records):
                
                taxonomy = record.annotations.get('taxonomy', [])
                organism = record.annotations.get('organism', "Unknown Organism")
                
                for i, taxon in enumerate(taxonomy):
                    col_name = f'taxon{i+1}'
                    result_df.at[result_df.index[start + idx], col_name] = taxon

                result_df.at[result_df.index[start + idx], 'organism'] = organism

    return result_df


In [ ]:
accession_work = accessions_df[0:20].copy()
res = get_batch_taxo_in_df(accession_work)
res2 = res.merge(df_with_accession[['accession', 'taxid']], on='accession', how='left')
res2

,accession,taxon1,taxon2,taxon3,taxon4,taxon5,taxon6,taxon7,organism,taxon8,taxid
0,NZ_LT667500,Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Erwiniaceae,Buchnera,Buchnera aphidicola,NaN,9
1,NZ_CP028435,Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Alteromonadales,Shewanellaceae,Shewanella,Shewanella putrefaciens,NaN,24
2,NZ_CP046329,Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Alteromonadales,Shewanellaceae,Shewanella,Shewanella putrefaciens,NaN,24
3,NZ_CP066369,Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Alteromonadales,Shewanellaceae,Shewanella,Shewanella putrefaciens,NaN,24
4,NZ_CP066370,Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Alteromonadales,Shewanellaceae,Shewanella,Shewanella putrefaciens,NaN,24
5,NZ_CP070865,Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Alteromonadales,Shewanellaceae,Shewanella,Shewanella putrefaciens,NaN,24
6,NZ_CP080635,Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Alteromonadales,Shewanellaceae,Shewanella,Shewanella putrefaciens,NaN,24
7,NZ_CP104755,Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Alteromonadales,Shewanellaceae,Shewanella,Shewanella putrefaciens,NaN,24
8,NZ_CP017169,Bacteria,Pseudomonadati,Myxococcota,Myxococcia,Myxococcales,Cystobacterineae,Myxococcaceae,Myxococcus xanthus,Myxococcus,34
9,NZ_CP017170,Bacteria,Pseudomonadati,Myxococcota,Myxococcia,Myxococcales,Cystobacterineae,Myxococcaceae,Myxococcus xanthus,Myxococcus,34


In [ ]:
# ACCESSION = "NZ_LT667500" RESULTAT METHODE I

with Entrez.efetch(db="nucleotide", id=['NZ_LT667500'], rettype="gb", retmode="text") as taxo_handle:
    x = SeqIO.read(taxo_handle, 'genbank')
    taxonomy = x.annotations['taxonomy']
    organism = x.annotations['organism']

order = next((e for e in taxonomy if e.endswith('ales')), None)

if order:
    regne, phylum = taxonomy[0], taxonomy[1]
    group = taxonomy[2] if len(taxonomy) > 2 and not taxonomy[2].endswith('ales') else taxonomy[1]
    
family, specie = organism.split(' ')[:2]
others = organism.split(' ')[2:]
TAXO_LEVELS = ["domain", "phylum", "group", "order", "family", "specie"] 
final_taxo = taxonomy[0], taxonomy[1], group, order, family, specie

print(*list(zip(TAXO_LEVELS, final_taxo)), sep='\n')


('domain', 'Bacteria')
('phylum', 'Pseudomonadati')
('group', 'Pseudomonadota')
('order', 'Enterobacterales')
('family', 'Buchnera')
('specie', 'aphidicola')


## Methode II : avec le taxid
En interogeant la db taxonomy
On obtient directement toute la lignée dans Lineage ou LineageEx



In [ ]:
handle = Entrez.efetch(db="taxonomy", id=str(9), retmode="xml")
records = Entrez.read(handle)
import json
print(json.dumps(records[0],indent=3))

{
   "TaxId": "9",
   "ScientificName": "Buchnera aphidicola",
   "OtherNames": {
      "Misspelling": [],
      "Anamorph": [],
      "Includes": [
         "Acyrthosiphon pisum symbiont P",
         "primary endosymbiont of Schizaphis graminum"
      ],
      "CommonName": [],
      "Synonym": [],
      "GenbankAnamorph": [],
      "GenbankSynonym": [],
      "Name": [
         {
            "ClassCDE": "authority",
            "DispName": "Buchnera aphidicola Munson et al. 1991"
         },
         {
            "ClassCDE": "type material",
            "DispName": "personal::Sg (ex Schizaphis graminum)"
         },
         {
            "ClassCDE": "type material",
            "DispName": "strain Sg (ex Schizaphis graminum)"
         }
      ],
      "Teleomorph": [],
      "Acronym": [],
      "Inpart": [],
      "Misnomer": [],
      "EquivalentName": []
   },
   "ParentTaxId": "32199",
   "Rank": "species",
   "Division": "Bacteria",
   "GeneticCode": {
      "GCId": "11",
    

## Méthode  II , via les  librairies ncbi_datasets sous console 

(ncbi_datasets) [hcourtei@ptb-jkp7wl3 (Fedora 40) ~]$ datasets summary taxonomy taxon 9 --as-json-lines | dataformat tsv taxonomy --template tax-summary | column -t


In [ ]:

Query  Taxid  Tax       name        Authority  Rank  Basionym  Basionym  authority  Curator  common          name      Has  type            material  Group           name  Superkingdom         name  Superkingdom      taxid  Kingdom      name     Kingdom   taxid  Phylum    name        Phylum  taxid  Class  name  Class  taxid  Order  name  Order  taxid  Family  name  Family  taxid  Genus  name  Genus  taxid  Species  name  Species  taxid  Scientific  name  is  formal
9      9      Buchnera  aphidicola  Munson     et    al.       1991      SPECIES    yes      enterobacteria  Bacteria  2    Pseudomonadati  3379134   Pseudomonadota  1224  Gammaproteobacteria  1236  Enterobacterales  91347  Erwiniaceae  1903409  Buchnera  32199  Buchnera  aphidicola  9       TRUE
